# Data failure RECOVER

import torch
from pathlib import Path

def checkpoint_epochs_table(root_dir, pattern="*A_to_B_model_latest.pt"):
    root = Path(root_dir)
    entries = []
    print(f"{'File Path':<80} {'Epoch'}")
    print("-" * 90)
    for path in root.rglob(pattern):
        if '0.5' not in str(path) and "NormalizeIntensityd" in str(path) and "latest-run" in str(path):
            try:
                ckpt = torch.load(path, map_location="cpu", weights_only=False)
                epoch = ckpt.get('epoch')
                print(f"{path}: {(epoch)}")
            except Exception as e:
                print(f"Warning: could not load {path}: {e}")
                epoch = None
            entries.append((str(path), epoch))

    # Sort by epoch (None values first)
    entries.sort(key=lambda x: (x[1] is None, x[1]))
        
    return entries


## Example usage:
entries = checkpoint_epochs_table('/projects/nian/synthrad2025/results/MC-IDDPM')


import os
import torch
root = "/projects/nian/synthrad2025/results/MC-IDDPM"
for file in os.listdir(root):
    if file != "wandb_old":
        latest_run = os.path.join(root, file, 'wandb', 'latest-run', 'files', 'model', 'A_to_B_model_latest.pt')
        if '0.5' not in latest_run and 'NormalizeIntensityd' in latest_run:
            ckpt = torch.load(latest_run, map_location="cpu", weights_only=False)
            epoch = ckpt.get('epoch')
            print(f"{latest_run}: {(epoch)}")

experiments_to_ignore = [
    "MC-IDDPM_Task1_2_5000_timestep_50_patchsize_2_SwinVIT_MSE_MAE_PSNR_128_128_32_NormalizeIntensityd_region_HN_TH_AB",
    "MC-IDDPM_Task1_2_5000_timestep_50_patchsize_2_SwinVIT_MSE_PSNR_128_128_32_NormalizeIntensityd_region_HN_TH_AB"
]

all_runs = {}
for path, epoch in entries:
    try:
        name = ''
        for i in range(1,7):
            name = f"{name}/{path.split('/')[i]}"
        if all_runs[name] < epoch:
            all_runs[name] = epoch
    except:
        name = ''
        for i in range(1,7):
            name = f"{name}/{path.split('/')[i]}"
        all_runs[name] = epoch
for key in all_runs.keys():
    print(f"{key}: {all_runs[key]}")


## Models missing
weights_they_say_missing = ["/projects/nian/synthrad2025/results/MC-IDDPM/MC-IDDPM_Task1_1_5000_timestep_50_patchsize_1_SwinVIT_MSE_256_256_32_NormalizeIntensityd_region_HN_TH_AB/wandb/run-20250519_173329-r67idn6s/files/model/A_to_B_model_latest.pt",
"/projects/nian/synthrad2025/results/MC-IDDPM/MC-IDDPM_Task1_2_5000_timestep_50_patchsize_2_SwinVIT_MSE_128_128_32_NormalizeIntensityd_region_HN_TH_AB/wandb/run-20250509_185037-gnlx5pz2/files/model/A_to_B_model_latest.pt",
"/projects/nian/synthrad2025/results/MC-IDDPM/MC-IDDPM_Task1_2_5000_timestep_50_patchsize_2_SwinVIT_MSE_MAE_128_128_32_NormalizeIntensityd_region_HN_TH_AB/wandb/run-20250519_171907-rgwpx8er/files/model/A_to_B_model_latest.pt",
"/projects/nian/synthrad2025/results/MC-IDDPM/MC-IDDPM_Task1_2_5000_timestep_50_patchsize_2_SwinVIT_MSE_MAE_PSNR_128_128_32_NormalizeIntensityd_region_HN_TH_AB/wandb/run-20250519_171850-la7t0mge/files/model/A_to_B_model_latest.pt",
"/projects/nian/synthrad2025/results/MC-IDDPM/MC-IDDPM_Task1_2_5000_timestep_50_patchsize_2_SwinVIT_MSE_MAE_PSNR_SSIM_128_128_32_NormalizeIntensityd_region_HN_TH_AB/wandb/run-20250519_190924-n2r4soe4/files/model/A_to_B_model_latest.pt",
"/projects/nian/synthrad2025/results/MC-IDDPM/MC-IDDPM_Task1_2_5000_timestep_50_patchsize_2_SwinVIT_MSE_MAE_SSIM_128_128_32_NormalizeIntensityd_region_HN_TH_AB/wandb/run-20250519_190937-nxq7dbq9/files/model/A_to_B_model_latest.pt",
"/projects/nian/synthrad2025/results/MC-IDDPM/MC-IDDPM_Task1_2_5000_timestep_50_patchsize_2_SwinVIT_MSE_PSNR_128_128_32_NormalizeIntensityd_region_HN_TH_AB/wandb/run-20250519_191011-amt6wieo/files/model/A_to_B_model_latest.pt",
"/projects/nian/synthrad2025/results/MC-IDDPM/MC-IDDPM_Task1_2_5000_timestep_50_patchsize_2_SwinVIT_MSE_SSIM_128_128_32_NormalizeIntensityd_region_HN_TH_AB/wandb/run-20250519_190938-cu291934/files/model/A_to_B_model_latest.pt"]

for path, epoch in entries:
    if path in weights_they_say_missing:
        print(f"{path:<80} {str(epoch)}")



# Code for evaluation of predictions. 
* Taken from https://github.com/SynthRAD2025/metrics?tab=readme-ov-file

In [1]:
import os
import json
from os import listdir
from os.path import isfile, join, isdir, exists
import SimpleITK as sitk
import numpy as np
from tqdm.auto import tqdm

import sys
sys.path.append("/projects/nian/synthrad2025/src/metrics/functions")
sys.path.append("/projects/nian/synthrad2025/src/metrics/evaluation")
from dose_metrics import DoseMetrics
from image_metrics import ImageMetrics
from segmentation_metrics import SegmentationMetrics
metrics = SegmentationMetrics()

## Move all regions to the save folder
/projects/nian/synthrad2025/experiments/MC-IDDPM/IDDPM/SwinVIT/registered/VS-DDPM_Task1_2_1000_timestep__patchsize_1_Unet_MAE_SSIM_tahn_AFP_128_128_32_distinctNorm_region_AB_linear_pen_var_random_T_CTminmax-1000_1600_finetune_new_NormalizeIntensityd_Scaledoverlap0.5_T_25_99/constant/AB

In [2]:
import shutil
regions = ['HN', 'TH']
base_name = "VS-DDPM_Task1_2_1000_timestep__patchsize_1_Unet_MAE_SSIM_tahn_AFP_128_128_32_distinctNorm_region_AB_linear_pen_var_random_T_CTminmax-1000_1600_finetune_new_NormalizeIntensityd_Scaledoverlap0.5_T_25_"
for i in range(99, 999, 100):
    complete_path = f"/projects/nian/synthrad2025/experiments/MC-IDDPM/IDDPM/SwinVIT/registered/{base_name}{i}/constant/AB"
    print(f"complete_path: {complete_path}")
    for region in regions:
        other_region = (f"/projects/nian/synthrad2025/experiments/MC-IDDPM/IDDPM/SwinVIT/registered/{base_name}{i}/constant/AB").replace('AB', region)
        destination = f"{complete_path.split('/AB')[0]}/{region}"
        print(f"destination: {destination}")
        #shutil.copytree(other_region, destination, dirs_exist_ok=True)
        print(f"other_region: {other_region}")
    


## Check which experiments are ready for inference

In [3]:
def find_epoch(data):
    if isinstance(data, dict):
        for key, value in data.items():
            if key == "epoch":
                return value
            found = find_epoch(value)
            if found is not None:
                return found
    elif isinstance(data, list):
        for item in data:
            found = find_epoch(item)
            if found is not None:
                return found
    return None

In [4]:
"""# OLD WAY
exps_folder = "/projects/nian/synthrad2025/experiments/MC-IDDPM/IDDPM/"
exp_to_ignore = ["wandb_old"]

training_cases = [
    "MC-IDDPM_Task1_2_5000_timestep_50_patchsize_2_SwinVIT_MSE_MAE_SSIM_DSC_128_128_32_NormalizeIntensityd_region_HN_TH_AB"
    ]

done_list = []
not_done_list = []
for experiment_name in listdir(exps_folder):
    if experiment_name in exp_to_ignore:
        continue
    flag_done = False
    experiment_path = join(exps_folder, experiment_name, "wandb")
    for exp_id in listdir(experiment_path):
        exp_id_path = join(experiment_path, exp_id, "files")
        wandb_summary = join(exp_id_path, 'wandb-summary.json')
        if isfile(wandb_summary):
            with open(wandb_summary, 'r') as file:
                data = json.load(file)
                epoch = find_epoch(data)
                if epoch==4999 or epoch==299:
                    done_list.append(experiment_name)
                    flag_done = True
                    break
    if not flag_done:
        not_done_list.append(experiment_name)
        

print("######## Ready for inference #########")
done_list.append("MC-IDDPM_Task1_1_5000_timestep_50_patchsize_1_SwinVIT_MSE_256_256_32_NormalizeIntensityd_region_HN_TH_AB")
for i in done_list:
    print(i)

print("######## Training #########")
Training_list = [

]

for i in Training_list:
    print(i) 
print("######## Not ready #########")
for i in not_done_list:
    if i not in Training_list:
        print(i)

"""

In [5]:
# NEW WAY
exps_folder = "/projects/nian/synthrad2025/experiments/MC-IDDPM/IDDPM/SwinVIT/registered"
exp_to_ignore = ["wandb_old",
    "MC-IDDPM_Task1_2_5000_timestep_50_patchsize_2_SwinVIT_MSE_DSC_128_128_32_NormalizeIntensityd_region_HN_TH_AB",
    "MC-IDDPM_Task1_2_5000_timestep_50_patchsize_2_SwinVIT_MSE_MAE_PSNR_SSIM_128_128_32_NormalizeIntensityd_region_HN_TH_AB",
    "MC-IDDPM_Task1_2_5000_timestep_50_patchsize_2_SwinVIT_MSE_MAE_PSNR_128_128_32_NormalizeIntensityd_region_HN_TH_AB",
    "MC-IDDPM_Task1_2_5000_timestep_50_patchsize_2_SwinVIT_MSE_PSNR_128_128_32_NormalizeIntensityd_region_HN_TH_AB"]


training_cases = [
    ]

cases_to_add = ["MC-IDDPM_Task1_2_1000_timestep_25_patchsize_2_SwinVIT_MSE_MAE_SSIM_128_128_32_distinctNorm_region_HN_TH_AB_DA_0.5new_NormalizeIntensityd_Scaled"]

done_list = cases_to_add
not_done_list = []
for experiment_name in listdir(exps_folder):
    if experiment_name in exp_to_ignore:
        continue
    flag_done = False
    experiment_path = join(exps_folder, experiment_name, "wandb")
    if experiment_name not in training_cases and "timestep_1000" not in experiment_name:
        done_list.append(experiment_name)
        flag_done = True
    if not flag_done:
        not_done_list.append(experiment_name)


# Include the experimenrts with pre-trained models
done_list.append("pre_trained_TotalSegmentator_region_HN_TH_AB_DA_0.5")
        

print("######## Ready for inference #########")
for i in done_list:
    print(i)

print("######## Training #########")
Training_list = [

]

for i in Training_list:
    print(i) 
print("######## Not ready #########")
for i in not_done_list:
    if i not in Training_list:
        print(i)



In [6]:
# Check which experiments are already infered
with open("/projects/nian/synthrad2025/Dataset/DataSet_Registered_2.0/Task1_data_split.json", 'r') as file:
    data_split = json.load(file) 

infer_path = "/projects/nian/synthrad2025/experiments/MC-IDDPM"
#dpm_methods = ["dpmsolver++", "karrasdpmsolver++", "karrassde-dpmsolver++", "sde-dpmsolver++"]
overlap_modes = ["constant", "gaussian"]

incomplete_list = []
complete_list = []
not_done_list = []

sampling_method_path = join(infer_path, "Pre_Trained") # experiments/MC-IDDPM/IDDPM
for exp_name in done_list: # Only consider the experiments ready for inference
    exp_path = join(sampling_method_path, exp_name) # experiments/MC-IDDPM/IDDPM/MC-IDDPM_Task1_2_5000_timestep_5_patchsize_2_SwinVIT_MSE_128_128_32_region_AB
    # In case the experiment folder exists in the sampling method
    if isdir(exp_path):
        # Check for each overlap_mode
        for overlap_mode in overlap_modes: # experiments/MC-IDDPM/IDDPM/MC-IDDPM_Task1_2_5000_timestep_5_patchsize_2_SwinVIT_MSE_128_128_32_region_AB/constant
            overlap_mode_path = join(exp_path, overlap_mode)
            if isdir(overlap_mode_path):
                expected_regions = exp_path.split('region_')[-1].split("_")
                for region in expected_regions: # experiments/MC-IDDPM/IDDPM/MC-IDDPM_Task1_2_5000_timestep_5_patchsize_2_SwinVIT_MSE_128_128_32_region_AB/constant/AB
                    region_path = join(overlap_mode_path, region)
                    if isdir(region_path):
                        if len(os.listdir(region_path)) < len(data_split[region]['val']):
                            #print(f"Some predictions are missing in {region_path}.\nPredicted={len(os.listdir(region_path))}, Expected={len(data_split[region]['val'])}")
                            incomplete_list.append(region_path)
                            continue
                        else:
                            complete_list.append(region_path)
                            continue
                    else:
                        not_done_list.append(region_path)
                        continue
            else:
                not_done_list.append(overlap_mode_path)
                continue
    else:
        not_done_list.append(exp_path)
        continue

sampling_method_path = join(infer_path, "IDDPM/SwinVIT/registered") # experiments/MC-IDDPM/IDDPM
for exp_name in done_list: # Only consider the experiments ready for inference
    exp_path = join(sampling_method_path, exp_name) # experiments/MC-IDDPM/IDDPM/MC-IDDPM_Task1_2_5000_timestep_5_patchsize_2_SwinVIT_MSE_128_128_32_region_AB
    print(f"exp_path: {exp_path}")
    
    # In case the experiment folder exists in the sampling method
    if isdir(exp_path):
        # Check for each overlap_mode
        for overlap_mode in overlap_modes: # experiments/MC-IDDPM/IDDPM/MC-IDDPM_Task1_2_5000_timestep_5_patchsize_2_SwinVIT_MSE_128_128_32_region_AB/constant
            overlap_mode_path = join(exp_path, overlap_mode)
            if isdir(overlap_mode_path):
                expected_regions = ["HN","TH","AB"] #exp_path.split('region_')[-1].split("_")
                for region in expected_regions: # experiments/MC-IDDPM/IDDPM/MC-IDDPM_Task1_2_5000_timestep_5_patchsize_2_SwinVIT_MSE_128_128_32_region_AB/constant/AB
                    print(f"region: {region}")
                    region_path = join(overlap_mode_path, region)
                    if isdir(region_path):
                        if len(os.listdir(region_path)) < len(data_split[region]['val']):
                            #print(f"Some predictions are missing in {region_path}.\nPredicted={len(os.listdir(region_path))}, Expected={len(data_split[region]['val'])}")
                            incomplete_list.append(region_path)
                            continue
                        else:
                            complete_list.append(region_path)
                            continue
                    else:
                        not_done_list.append(region_path)
                        continue
            else:
                not_done_list.append(overlap_mode_path)
                continue
    else:
        not_done_list.append(exp_path)
        continue

sampling_method_path = join(infer_path, "IDDPM") # experiments/MC-IDDPM/IDDPM
for exp_name in listdir(sampling_method_path): # Only consider the experiments ready for inference
    exp_path = join(sampling_method_path, exp_name) # experiments/MC-IDDPM/IDDPM/MC-IDDPM_Task1_2_5000_timestep_5_patchsize_2_SwinVIT_MSE_128_128_32_region_AB
    # In case the experiment folder exists in the sampling method
    if isdir(exp_path):
        # Check for each overlap_mode
        for overlap_mode in overlap_modes: # experiments/MC-IDDPM/IDDPM/MC-IDDPM_Task1_2_5000_timestep_5_patchsize_2_SwinVIT_MSE_128_128_32_region_AB/constant
            overlap_mode_path = join(exp_path, overlap_mode)
            if isdir(overlap_mode_path):
                expected_regions = exp_path.split('region_')[-1].split("_")
                for region in expected_regions: # experiments/MC-IDDPM/IDDPM/MC-IDDPM_Task1_2_5000_timestep_5_patchsize_2_SwinVIT_MSE_128_128_32_region_AB/constant/AB
                    region_path = join(overlap_mode_path, region)
                    if isdir(region_path):
                        try:
                            if len(os.listdir(region_path)) < len(data_split[region]['val']):
                                #print(f"Some predictions are missing in {region_path}.\nPredicted={len(os.listdir(region_path))}, Expected={len(data_split[region]['val'])}")
                                incomplete_list.append(region_path)
                                continue
                            else:
                                complete_list.append(region_path)
                                continue
                        except:
                            print(f"region_path: {region_path}")
                            incomplete_list.append(region_path)
                            continue
                    else:
                        not_done_list.append(region_path)
                        continue
            else:
                not_done_list.append(overlap_mode_path)
                continue
    else:
        not_done_list.append(exp_path)
        continue
"""
sampling_method_path = join(infer_path, "DPM") # experiments/MC-IDDPM/DPM
for exp_name in done_list: # Only consider the experiments ready for inference
    exp_path = join(sampling_method_path, exp_name) # experiments/MC-IDDPM/DPM/MC-IDDPM_Task1_2_5000_timestep_5_patchsize_2_SwinVIT_MSE_128_128_32_region_AB
    # In case the experiment folder exists in the sampling method
    if isdir(exp_path):
        # Check each dpm_methods:
        for dpm_method in dpm_methods:
            dpm_method_path = join(exp_path, dpm_method) # experiments/MC-IDDPM/DPM/MC-IDDPM_Task1_2_5000_timestep_5_patchsize_2_SwinVIT_MSE_128_128_32_region_AB/dpmsolver++
            if isdir(dpm_method_path):
                # Check for each overlap_mode
                for overlap_mode in overlap_modes: # experiments/MC-IDDPM/DPM/MC-IDDPM_Task1_2_5000_timestep_5_patchsize_2_SwinVIT_MSE_128_128_32_region_AB/dpmsolver++/constant
                    overlap_mode_path = join(dpm_method_path, overlap_mode)
                    if isdir(overlap_mode_path):
                        expected_regions = exp_path.split('region_')[-1].split("_")
                        for region in expected_regions: # experiments/MC-IDDPM/DPM/MC-IDDPM_Task1_2_5000_timestep_5_patchsize_2_SwinVIT_MSE_128_128_32_region_AB/dpmsolver++/constant/AB
                            region_path = join(overlap_mode_path, region)
                            if isdir(region_path):
                                if len(os.listdir(region_path)) < len(data_split[region]['val']):
                                    #print(f"Some predictions are missing in {region_path}.\nPredicted={len(os.listdir(region_path))}, Expected={len(data_split[region]['val'])}")
                                    incomplete_list.append(region_path)
                                    continue
                                else:
                                    complete_list.append(region_path)
                                    continue
                            else:
                                not_done_list.append(region_path)
                                continue
                    else:
                        not_done_list.append(overlap_mode_path)
                        continue
            else:
                not_done_list.append(dpm_method_path)
                continue
    else:
        not_done_list.append(exp_path)
        continue
"""
#print("######## COMPLETE #########")
#for i in complete_list:
#    #print(i)


print("######## INCOMPLETE #########")
for i in incomplete_list:
    region = i[-2:]
    #if len(os.listdir(i)) < len(data_split[region]['val']):
    print(i)
    #else:
    #    print("WRONG")

print("######## DOING #########")
infering_list = []

doing_list = []
for i in not_done_list:
    if "gaussian" in i:
        if i.split("gaussian")[0] in infering_list:
            print(i)
            doing_list.append(i)
    elif "constant" in i:
        if i.split("constant")[0] in infering_list:
            print(i)
        doing_list.append(i)
    #for dpm_method in dpm_methods:
    #    if dpm_method in i:
    #        if i.split(dpm_method)[0] in infering_list:
    #            print(i)
    #            doing_list.append(i)

print("######## NOT DONE #########")
for i in not_done_list:
    if i not in doing_list:
        print(i)

print("######## Ready to compute metrics #########")
ready_to_compute_list = []
for i in complete_list:
    if not isfile(join(i, "similarity.json")) or not isfile(join(i, "consistency.json")):
        ready_to_compute_list.append(i)
        print(i)


## Functions

In [7]:
def convert_nii_to_mha(input_path, output_path):
    """
    Convert a .nii.gz file to .mha format.

    Parameters:
        input_path (str): Path to the input .nii.gz file.
        output_path (str): Path to save the output .mha file. If None, the output file will be saved 
                           in the same directory as the input file with a .mha extension.
    """
    image = sitk.ReadImage(input_path)
    if output_path is None:
        output_path = os.path.splitext(os.path.splitext(input_path)[0])[0] + '.mha'
    sitk.WriteImage(image, output_path)
    print(f"Converted: {input_path} -> {output_path}")

def convert_folder_nii_to_mha(input_folder, output_folder):
    """
    Convert all .nii.gz files in a folder structure to .mha format and save them in the specified output folder.

    Parameters:
        input_folder (str): Path to the input folder containing .nii.gz files organized in subdirectories.
        output_folder (str): Path to the output folder where converted .mha files will be saved, 
                             maintaining the same folder structure as the input.
    """
    for region in listdir(input_folder):
        region_path = join(input_folder, region)
        for patient_id in listdir(region_path):
            patient_path = join(region_path, patient_id)
            if not isdir(patient_path):
                continue
            for file in listdir(patient_path):
                if file.endswith(".nii.gz"):
                    input_path = join(patient_path, file)
                    output_path = join(output_folder, region, patient_id, file[:-7] + ".mha")
                    print(f"Converting: {input_path} -> {output_path}")
                    convert_nii_to_mha(input_path, output_path)

if False:
    # Execute the conversion
    input_folder = '/your/input/folder/path'
    output_folder = '/your/output/folder/path'
    convert_folder_nii_to_mha(input_folder, output_folder)

In [8]:
def load_mha_to_np(file_path):
    """
    Load a .mha file and convert it to a NumPy array.

    Parameters:
        file_path (str): Path to the .mha file.

    Returns:
        np.ndarray: The image data as a NumPy array.
    """
    image = sitk.ReadImage(file_path)
    return np.transpose(sitk.GetArrayFromImage(image), (2, 1, 0))

In [9]:
def save_list_to_json(data_list, output_path):
    """
    Save a list of dictionaries to a JSON file.

    Parameters:
        data_list (list): The list of dictionaries to save. Each dictionary represents a case with metrics and file paths.
        output_path (str): The path to the output JSON file.

    Returns:
        None
    """
    mean_dict = get_mean_dict(data_list)

    save_dict = {
        "mean": mean_dict,
        "cases": data_list
    }
    with open(output_path, "w") as f:
        json.dump(save_dict, f, indent=4)
    print(f"Data saved to {output_path}")

def get_mean_dict(data_list):    
    """
    Calculate the mean values for numeric keys in a list of dictionaries.

    Parameters:
        data_list (list): The list of dictionaries containing numeric values.

    Returns:
        dict: A dictionary with the mean values for each numeric key.
    """
    mean_dict = {}

    keys_to_consider = []
    for key, value in data_list[0].items():
        if isinstance(value, (int, float)):
            keys_to_consider.append(key)

    for key_name in keys_to_consider:
        all_cases_value = []
        for entry in data_list:
            case_value = entry[key_name]
            all_cases_value.append(case_value)
        mean_dict[key_name] = sum(all_cases_value)/len(all_cases_value)
    return mean_dict

## Image similarity

In [10]:
def score_patient_loop(dataset_path_A_B_C, dataset_path_D, input_folder): 
    """
    Evaluate the similarity metrics for predicted .mha files against ground truth .mha files.

    Parameters:
        dataset_path_A_B_C (str): Path to the dataset containing regions and patient folders (A, B, C).
        dataset_path_D (str): Path to the fallback dataset containing regions and patient folders (D).
        input_folder (str): Path to the folder containing predicted .mha files organized by regions and patients.

    Returns:
        list: A list of dictionaries containing similarity metrics for each patient, 
              along with paths to the predicted and ground truth files.
    """
    metrics = ImageMetrics()
    score_patients_L = [
        # {'mae': 105.05763746533839, 
        # 'psnr': 26.016467248144266, 
        # 'ms_ssim': 0.9101366114238764,
        # 'pred': '/projects/nian/synthrad2025/experiments/MC-IDDPM/MC-IDDPM_Task1_2_5000_timestep_5_patchsize_2_SwinVIT_MSE_128_128_32_region_HN_TH_AB/HN/1HND004/ct_pred.mha',
        # 'gt': '/projects/nian/synthrad2025/Dataset/DataSet_Registered_2.0/synthRAD2025_Task1_Train_D/Task1/HN/1HND004/ct.mha'
        #}
    ]
    with open("/projects/nian/synthrad2025/Dataset/DataSet_Registered_2.0/Task1_data_split.json", 'r') as file:
        data_split = json.load(file) 
    
    region = input_folder[-2:]
    print(f"Predicting region: {region}")
    region_path = input_folder
    if len(os.listdir(region_path)) < len(data_split[region]['val']):
        raise ValueError(f"Some predictions are missing in {region_path}.\nPredicted={len(os.listdir(region_path))}, Expected={len(data_split[region]['val'])}")
    if not isdir(region_path):
        raise ValueError(f"Is not dir: {region_path}")
    for patient_id in tqdm(os.listdir(region_path), desc="Processing patient"):
        patient_path = join(region_path, patient_id)
        if not isdir(patient_path):
            print(f"Not dir: {patient_path}")
            continue
        
        if len(os.listdir(patient_path))==0:
            raise ValueError(f"Patient doesn't have prediction: {patient_path}")
            
        for file in os.listdir(patient_path):
            if file.endswith(".mha"):

                predicted_path = join(patient_path, file)
                
                # Try dataset_path_A_B_C first
                gt_path_A = join(dataset_path_A_B_C, region, patient_id, 'ct.mha')
                mask_path_A = join(dataset_path_A_B_C, region, patient_id, 'mask.mha')
                
                # Fallback to dataset_path_D if not found
                gt_path_D = join(dataset_path_D, region, patient_id, 'ct.mha')
                mask_path_D = join(dataset_path_D, region, patient_id, 'mask.mha')
                
                ground_truth_path = gt_path_A if isfile(gt_path_A) else gt_path_D
                mask_path = mask_path_A if isfile(mask_path_A) else mask_path_D

                print(f"ground_truth_path: {ground_truth_path}")
                print(f"mask_path: {mask_path}")
                print(f"predicted_path: {predicted_path}")
                
                results = metrics.score_patient(
                    gt_img=load_mha_to_np(ground_truth_path), 
                    synthetic_ct=load_mha_to_np(predicted_path), 
                    mask=load_mha_to_np(mask_path)
                )
                results['pred'] = predicted_path
                results['gt'] = ground_truth_path

                score_patients_L.append(results)
            
    return score_patients_L

In [11]:
dataset_path_A_B_C = "/projects/nian/synthrad2025/Dataset/DataSet_Registered_2.0/synthRAD2025_Task1_Train/Task1"
dataset_path_D = "/projects/nian/synthrad2025/Dataset/DataSet_Registered_2.0/synthRAD2025_Task1_Train_D/Task1"

score_patients_total = []
for input_folder in ready_to_compute_list:
    
    #print(f"Doing {input_folder}")
    output_path = join(input_folder, 'similarity.json')
    if isfile(output_path):
        print(f"{output_path} already predicted.")
        continue
    if not isdir(input_folder):
        
        print(f"{input_folder} does not exist")
        continue
    else:
        print(f"Predicting {output_path}")
        score_patients_L = score_patient_loop(
            dataset_path_A_B_C=dataset_path_A_B_C, 
            dataset_path_D=dataset_path_D, 
            input_folder=input_folder
            )
        print(f"score_patients_L: {score_patients_L}")
        save_list_to_json(data_list=score_patients_L, output_path=output_path)
        for i in score_patients_L:
            score_patients_total.append(i)

In [12]:
### Compute mean for splited regions
root_folder = "/projects/nian/synthrad2025/experiments/MC-IDDPM/IDDPM/SwinVIT/registered"
input_folders_list = listdir(root_folder)
overlap_modes = ["constant", "gaussian"]
region_list = ['HN', 'TH', 'AB']

for overlap_mode in overlap_modes:
    for input_folder in input_folders_list:
        score_patients_total = []
        save_flag = True
        # Check if it is per region training
        regions = input_folder.split('_region_')[-1].split('_')
        print(f"regions: {regions}")
        if len(regions)==1:
            input_folder = input_folder[:-3]
            for region in region_list:
                here_input_folder = f"{input_folder}_{region}"
                json_file = join(root_folder, here_input_folder, overlap_mode, region, 'similarity.json')
                output_path = join(root_folder, here_input_folder, overlap_mode, region, 'total_similarity.json')
                if isfile(json_file):
                    with open(json_file, 'r') as file:
                        score_patients_L = json.load(file)
                        score_patients_L = score_patients_L["cases"]
                    for i in score_patients_L:
                        score_patients_total.append(i)
                else:
                    print(f"Does not exist: {json_file}")
                    save_flag = False
                    break
        # In case all prediction are in the same experiment folder
        else:
            for region in region_list:
                here_input_folder = input_folder
                json_file = join(root_folder, here_input_folder, overlap_mode, region, 'similarity.json')
                output_path = join(root_folder, here_input_folder, overlap_mode, region, 'total_similarity.json')
                if isfile(json_file):
                    with open(json_file, 'r') as file:
                        score_patients_L = json.load(file)
                        score_patients_L = score_patients_L["cases"]
                    for i in score_patients_L:
                        score_patients_total.append(i)
                else:
                    print(f"Does not exist: {json_file}")
                    save_flag = False
                    break
        if save_flag:
            save_list_to_json(data_list=score_patients_total, output_path=output_path)

In [13]:
### Compute mean for splited regions
root_folder = "/projects/nian/synthrad2025/experiments/MC-IDDPM/DPM"
input_folders_list = listdir(root_folder)
overlap_modes = ["constant", "gaussian"]
region_list = ['HN', 'TH', 'AB']

for overlap_mode in overlap_modes:
    for input_folder in input_folders_list:
        for sampling_mode in listdir( join(root_folder,input_folder)):
            score_patients_total = []
            save_flag = True
            # Check if it is per region training
            regions = input_folder.split('_region_')[-1].split('_')
            if len(regions)==1:
                input_folder = input_folder[:-3]
                for region in region_list:
                    here_input_folder = f"{input_folder}_{region}"
                    json_file = join(root_folder, here_input_folder, sampling_mode,  overlap_mode, region, 'similarity.json')
                    output_path = join(root_folder, here_input_folder, sampling_mode, overlap_mode, region, 'total_similarity.json')
                    if isfile(json_file):
                        with open(json_file, 'r') as file:
                            score_patients_L = json.load(file)
                            score_patients_L = score_patients_L["cases"]
                        for i in score_patients_L:
                            score_patients_total.append(i)
                    else:
                        print(f"Does not exist: {json_file}")
                        save_flag = False
                        break
            # In case all prediction are in the same experiment folder
            elif len(regions)==3:
                for region in region_list:
                    here_input_folder = input_folder
                    json_file = join(root_folder, here_input_folder, sampling_mode, overlap_mode, region, 'similarity.json')
                    output_path = join(root_folder, here_input_folder, sampling_mode, overlap_mode, region, 'total_similarity.json')
                    if isfile(json_file):
                        with open(json_file, 'r') as file:
                            score_patients_L = json.load(file)
                            score_patients_L = score_patients_L["cases"]
                        for i in score_patients_L:
                            score_patients_total.append(i)
                    else:
                        print(f"Does not exist: {json_file}")
                        save_flag = False
                        break
            else:
                print(f"Not all regions {input_folder}")
                continue
            if save_flag:
                save_list_to_json(data_list=score_patients_total, output_path=output_path)


In [14]:
### Compute mean for splited regions
root_folders = ["/projects/nian/synthrad2025/experiments/MC-IDDPM/IDDPM/SwinVIT/registered"]

overlap_modes = ["constant", "gaussian"]
region_list = ['HN', 'TH', 'AB']

for root_folder in root_folders:
    for overlap_mode in overlap_modes:
        score_patients_total = []
        save_flag = True
        # Check if it is per region training
        regions = input_folder.split('_region_')[-1].split('_')
       
        input_folder = input_folder[:-3]
        for region in region_list:
            here_input_folder = f"{input_folder}_{region}"

            json_file = join(root_folder, overlap_mode, region, 'similarity.json')
            output_path = join(root_folder, overlap_mode, region, 'total_similarity.json')
            if isfile(json_file):
                with open(json_file, 'r') as file:
                    score_patients_L = json.load(file)
                    score_patients_L = score_patients_L["cases"]
                for i in score_patients_L:
                    score_patients_total.append(i)
            else:
                print(f"Does not exist: {json_file}")
                save_flag = False
                break
        
        if save_flag:
            save_list_to_json(data_list=score_patients_total, output_path=output_path)


## Geometry consistency

In [15]:
def score_seg_patient_loop(gt_seg_path, input_folder): 
    """
    Evaluate the similarity metrics for predicted .mha files against ground truth .mha files.

    Parameters:
        dataset_path_A_B_C (str): Path to the dataset containing regions and patient folders (A, B, C).
        dataset_path_D (str): Path to the fallback dataset containing regions and patient folders (D).
        input_folder (str): Path to the folder containing predicted .mha files organized by regions and patients.

    Returns:
        list: A list of dictionaries containing similarity metrics for each patient, 
              along with paths to the predicted and ground truth files.
    """
    metrics = SegmentationMetrics()
    score_patients_L = []

    with open("/projects/nian/synthrad2025/Dataset/DataSet_Registered_2.0/Task1_data_split.json", 'r') as file:
        data_split = json.load(file) 
        
    
    region = input_folder[-2:]
    print(f"Predicting region: {region}")
    region_path = input_folder
    if len(os.listdir(region_path)) < len(data_split[region]['val']):
        raise ValueError(f"Some predictions are missing in {region_path}.\nPredicted={len(os.listdir(region_path))}, Expected={len(data_split[region]['val'])}")
    if not isdir(region_path):
        raise ValueError(f"Is not dir: {region_path}")
    for patient_id in tqdm(os.listdir(region_path), desc="Processing patient"):
        patient_path = join(region_path, patient_id)
        if not isdir(patient_path):
            continue
        predicted_path = join(patient_path,'ct_pred.mha')
        if not exists(predicted_path):
            print(f"{predicted_path} does not exist.")
        gt_path = join(gt_seg_path, region, patient_id, 'pred_seg.mha')
        
        results = metrics.score_patient(
            gt_segmentation=load_mha_to_np(gt_path), 
            synthetic_ct_location=predicted_path, 
            mask=None,
            patient_id=patient_id
        )
        
        results['pred'] = predicted_path
        results['pred_seg'] = f"{predicted_path.split('/ct_pred.mha')[0]}/pred_seg.mha"
        results['gt_seg'] = gt_path
        print(results)
        score_patients_L.append(results)
    return score_patients_L

def create_gt_seg(dataset_path_A_B_C, dataset_path_D, out_path):
    """
    Generate ground truth segmentation files for all patients in the dataset.

    Parameters:
        dataset_path_A_B_C (str): Path to the dataset containing regions and patient folders (A, B, C).
        dataset_path_D (str): Path to the fallback dataset containing regions and patient folders (D).
        out_path (str): Path to the output folder where the generated segmentation files will be saved.

    Returns:
        None
    """
    metrics = SegmentationMetrics()
    for source in [dataset_path_A_B_C, dataset_path_D]:
        for region in listdir(join(source)):
            region_path = join(source, region)
            for patient_id in listdir(region_path):
                if patient_id == "overviews":
                    continue
                patient_id_path = join(region_path, patient_id)
                patient_id_ct_path = join(patient_id_path, "ct.mha")
                # Set output path and create folder
                out_seg_path = join(out_path, region, patient_id)
                os.makedirs(out_seg_path, exist_ok=True)
                metrics.pred_gt(
                    in_ct_path=patient_id_ct_path, 
                    out_seg_path=out_seg_path
                )

In [16]:
dataset_path_A_B_C = "/projects/nian/synthrad2025/Dataset/DataSet_Registered_2.0/synthRAD2025_Task1_Train/Task1"
dataset_path_D = "/projects/nian/synthrad2025/Dataset/DataSet_Registered_2.0/synthRAD2025_Task1_Train_D/Task1" 
out_path = "/projects/nian/synthrad2025/Dataset/DataSet_Registered_2.0/Task1_seg"
if False:
    create_gt_seg(dataset_path_A_B_C, dataset_path_D, out_path)

In [17]:
gt_seg_path = "/projects/nian/synthrad2025/Dataset/DataSet_Registered_2.0/Task1_seg/Task1"

score_patients_total = []
for input_folder in ready_to_compute_list:
    output_path = join(input_folder, 'consistency.json')
    if isfile(output_path):
        print(f"{output_path} already predicted.")
        continue 
    if not isdir(input_folder):
        print(f"{input_folder} does not exist")
        continue
    else:
        score_patients_L = score_seg_patient_loop(
            gt_seg_path=gt_seg_path,
            input_folder=input_folder
            )
        save_list_to_json(data_list=score_patients_L, output_path=output_path)
        for i in score_patients_L:
            score_patients_total.append(i)


In [18]:
### Compute mean for splited regions
root_folder = "/projects/nian/synthrad2025/experiments/MC-IDDPM/IDDPM/SwinVIT/registered"
input_folders_list = listdir(root_folder)
overlap_modes = ["constant", "gaussian"]
region_list = ['HN', 'TH', 'AB']

for overlap_mode in overlap_modes:
    for input_folder in input_folders_list:
        score_patients_total = []
        save_flag = True
        # Check if it is per region training
        regions = input_folder.split('_region_')[-1].split('_')
        if len(regions)==1:
            input_folder = input_folder[:-3]
            for region in region_list:
                here_input_folder = f"{input_folder}_{region}"

                json_file = join(root_folder, here_input_folder, overlap_mode, region, 'consistency.json')
                output_path = join(root_folder, here_input_folder, overlap_mode, region, 'total_consistency.json')
                if isfile(json_file):
                    with open(json_file, 'r') as file:
                        score_patients_L = json.load(file)
                        score_patients_L = score_patients_L["cases"]
                    for i in score_patients_L:
                        score_patients_total.append(i)
                else:
                    print(f"Does not exist: {json_file}")
                    save_flag = False
                    break
        # In case all prediction are in the same experiment folder
        else:
            for region in region_list:
                here_input_folder = input_folder
                json_file = join(root_folder, here_input_folder, overlap_mode, region, 'consistency.json')
                output_path = join(root_folder, here_input_folder, overlap_mode, region, 'total_consistency.json')
                if isfile(json_file):
                    with open(json_file, 'r') as file:
                        score_patients_L = json.load(file)
                        score_patients_L = score_patients_L["cases"]
                    for i in score_patients_L:
                        score_patients_total.append(i)
                else:
                    print(f"Does not exist: {json_file}")
                    save_flag = False
                    break
        if save_flag:
            save_list_to_json(data_list=score_patients_total, output_path=output_path)


In [19]:
### Compute mean for splited regions
root_folders = ["/projects/nian/synthrad2025/experiments/MC-IDDPM/Pre_Trained/pre_trained_TotalSegmentator_region_HN_TH_AB_DA_0.5"]

overlap_modes = ["constant", "gaussian"]
region_list = ['HN', 'TH', 'AB']

for root_folder in root_folders:
    for overlap_mode in overlap_modes:
        score_patients_total = []
        save_flag = True
        # Check if it is per region training
        regions = input_folder.split('_region_')[-1].split('_')
       
        input_folder = input_folder[:-3]
        for region in region_list:
            here_input_folder = f"{input_folder}_{region}"

            json_file = join(root_folder, overlap_mode, region, 'consistency.json')
            output_path = join(root_folder, overlap_mode, region, 'total_consistency.json')
            if isfile(json_file):
                with open(json_file, 'r') as file:
                    score_patients_L = json.load(file)
                    score_patients_L = score_patients_L["cases"]
                for i in score_patients_L:
                    score_patients_total.append(i)
            else:
                print(f"Does not exist: {json_file}")
                save_flag = False
                break
        
        if save_flag:
            save_list_to_json(data_list=score_patients_total, output_path=output_path)


## Dose evaluation
* No idea how to do it :D

In [20]:
"""dose_path = 'path/to/treatment_plans'
predicted_path = "path/to/prediction.mha"
patient_id="1BA000"

metrics = DoseMetrics(dose_path)
print(metrics.score_patient(patient_id, predicted_path))"""

## All results

In [21]:
import pandas as pd
import os
def find_all_files(start_dirs, file_name):
    matches = []
    for start_dir in start_dirs:
        for root, dirs, files in os.walk(start_dir):
            if file_name in files:
                full_path = join(root, file_name)
                matches.append(full_path)
    return matches

In [22]:
place_to_search = ["/projects/nian/synthrad2025/experiments/MC-IDDPM/IDDPM/SwinVIT/registered"]

consistency_paths = find_all_files(
    start_dirs=place_to_search,
    file_name='consistency.json')

similarity_paths = find_all_files(
    start_dirs=place_to_search,
    file_name='similarity.json')


In [23]:
import numpy as np
rows = []
for file_path in consistency_paths:
    exp_name = file_path.split("/")[7]
    region_name = file_path.split('/')[-2]
    overlap_name = file_path.split('/')[-3]
    sampling_method = file_path.split('/')[6]
    sampling_method_2 = file_path.split('/')[8]
    
    if "++" in sampling_method_2:
        sampling_method_2 = "_"+sampling_method_2
    else:
        sampling_method_2 = ''
        
    similarity_path = f"{os.path.dirname(file_path)}/similarity.json"
    with open(similarity_path, 'r') as f:
        data = json.load(f)
        mean = data['mean']
        mae = mean['mae']
        psnr = mean['psnr']
        ms_ssim = mean['ms_ssim']
        if np.isnan(mae):
            print(f"file_path: {file_path}")
        
    
    with open(file_path, 'r') as f:
        data = json.load(f)
        mean = data['mean']
        dsc = mean['DICE']
        hd95 = mean['HD95']

    rows.append({
        'exp_name':   exp_name,
        'sampling_method': f"{sampling_method}{sampling_method_2}",
        'overlap': overlap_name,
        'region': region_name,
        'MAE':        mae,
        'PSNR':       psnr,
        'MS_SSIM':    ms_ssim,
        'DICE':       dsc,
        'HD95':       hd95,
    })
df = pd.DataFrame(rows)
df = df.sort_values(by=['exp_name', 'region', 'MAE'])
pd.set_option('display.float_format', '{:.7f}'.format)
# Show all rows and prevent column content truncation
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)

# Optional: show all columns too
pd.set_option('display.max_columns', None)

# Optional: expand column width to match notebook size
pd.set_option('display.width', 0)

In [24]:
total_consistency_paths = find_all_files(
    start_dirs=place_to_search,
    file_name='total_consistency.json')

total_similarity_paths = find_all_files(
    start_dirs=place_to_search,
    file_name='total_similarity.json')
print(total_similarity_paths)

In [25]:

rows = []

for file_path in total_consistency_paths:
    print(file_path)
    exp_name = file_path.split("/")[-4]
    region_name = file_path.split('/')[-2]
    overlap_name = file_path.split('/')[-3]
    sampling_method = file_path.split('/')[6]
    sampling_method_2 = file_path.split('/')[8]
    if "++" in sampling_method_2:
        check_exp_name = exp_name+"/"+sampling_method_2
        sampling_method_2 = "_"+sampling_method_2
    else:
        sampling_method_2 = ''
        check_exp_name = exp_name

    total_similarity_path = f"{os.path.dirname(file_path)}/total_similarity.json"
    with open(total_similarity_path, 'r') as f:
        data = json.load(f)
        mean = data['mean']
        mae = mean['mae']
        psnr = mean['psnr']
        ms_ssim = mean['ms_ssim']
    
    with open(file_path, 'r') as f:
        data = json.load(f)
        mean = data['mean']
        dsc = mean['DICE']
        hd95 = mean['HD95']

    rows.append({
        'exp_name':   exp_name,
        #'sampling_method': f"{sampling_method}{sampling_method_2}",
        'overlap': overlap_name,
        'region': region_name,
        'MAE':        mae,
        'PSNR':       psnr,
        'MS_SSIM':    ms_ssim,
        'DICE':       dsc,
        'HD95':       hd95,
    })
    



In [26]:
import openpyxl
df = pd.DataFrame(rows)
#df = df.sort_values(by=['exp_name', 'region', 'MAE',])
df = df.sort_values(by=['MAE'], ascending=True)
pd.set_option('display.float_format', '{:.7f}'.format)
# Show all rows and prevent column content truncation
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)

# Optional: show all columns too
pd.set_option('display.max_columns', None)

# Optional: expand column width to match notebook size
pd.set_option('display.width', 0)
df.to_excel("/projects/nian/synthrad2025/experiments/MC-IDDPM/results.xlsx", index=False)
df



In [27]:
err

In [ ]:

import os
import zipfile

def zip_folder(folder_path, output_path):
    with zipfile.ZipFile(output_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for root, dirs, files in os.walk(folder_path):
            for file in files:
                full_path = os.path.join(root, file)
                # Store the file relative to the folder being zipped
                rel_path = os.path.relpath(full_path, start=folder_path)
                zipf.write(full_path, arcname=rel_path)

# Example usage
zip_folder(
    '/projects/nian/synthrad2025/Test_results/VS-DDPM_Task1_2_1000_timestep__patchsize_2_Unet_MAE_SSIM_tahn_AFP_128_128_32_distinctNorm_region_HN_TH_AB_linear_pen_var_random_T_CTminmax-1000_1600_finetune_adjustStep_adjustOverlap/constant/', 
    '/projects/nian/synthrad2025/Test_results/VS-DDPM_Task1_2_1000_timestep__patchsize_2_Unet_MAE_SSIM_tahn_AFP_128_128_32_distinctNorm_region_HN_TH_AB_linear_pen_var_random_T_CTminmax-1000_1600_finetune_adjustStep_adjustOverlap/constant.zip')


In [ ]:
root_path = "/projects/nian/synthrad2025/experiments/MC-IDDPM/IDDPM/SwinVIT/registered"
from os import listdir
from os.path import join
for exp in listdir(root_path):
    exp_path = join(root_path, exp, 'constant', 'AB')


    if os.path.exists(join(exp_path, "total_consistency.json")):
        pass
    else:
        print(f"Not complete: {exp_path}")


    if os.path.exists(join(exp_path, "total_similarity.json")):
        pass
    else:
        print(f"Not complete: {exp_path}")


In [ ]:
import pandas as pd

# Define the name of your input file
input_file = 'results.xlsx - Sheet1.csv'

# Read the data from the CSV file
try:
    df = pd.read_csv(input_file)

    # Create a new DataFrame to store the experiment names and their ranks
    ranked_df = pd.DataFrame()
    ranked_df['exp_name'] = df['exp_name']

    # Rank the metrics.
    # For MAE and HD95, lower values are better (rank 1), so ascending=True.
    # For PSNR, MS_SSIM, and DICE, higher values are better, so ascending=False.
    
    ranked_df['MAE_Rank'] = df['MAE'].rank(ascending=True)
    ranked_df['HD95_Rank'] = df['HD95'].rank(ascending=True)
    
    ranked_df['PSNR_Rank'] = df['PSNR'].rank(ascending=False)
    ranked_df['MS_SSIM_Rank'] = df['MS_SSIM'].rank(ascending=False)
    ranked_df['DICE_Rank'] = df['DICE'].rank(ascending=False)

    # Reorder columns for clarity
    ranked_df = ranked_df[[
        'exp_name',
        'MAE_Rank',
        'PSNR_Rank',
        'MS_SSIM_Rank',
        'DICE_Rank',
        'HD95_Rank'
    ]]

    # Define the name for the output file
    output_file = 'experiment_rankings.csv'
    
    # Save the ranked DataFrame to a new CSV file
    ranked_df.to_csv(output_file, index=False)

    print(f"Successfully created rankings file: {output_file}")
    print("\nHere's a preview of the rankings:")
    print(ranked_df.head())

except FileNotFoundError:
    print(f"Error: The file '{input_file}' was not found.")
except Exception as e:
    print(f"An error occurred: {e}")